# Pipeline completo (CORRETO) — IDS multiclasse SOME/IP (`content_ext`)

Ponta a ponta, com o **split correto (temporal por arquivo)** — sem vazamento temporal:
**download dos PCAPs → extração de features → split temporal → treino → teste →
métricas + matriz de confusão + curvas**.

**Por que temporal e não aleatório?** Tráfego é sequencial; um split aleatório coloca pacotes da
mesma rajada de ataque em treino e teste, inflando as métricas (*data leakage* temporal). Aqui
treinamos nos **primeiros 70% (cronológicos) de cada PCAP** e testamos nos **30% finais**.
Ver `docs/relatorio-validacao-split.md` e a comparação em `02-comparacao-splits.ipynb`.

Interruptor **`FROM_SCRATCH`**: `False` (padrão) usa as features versionadas (LFS, rápido);
`True` baixa os PCAPs (~1,9 GB) e **re-extrai do zero** (~6 min).

In [ ]:
# 0) Setup — Colab: clona o repo + baixa features (LFS, blindado). Local: assume raiz do repo.
import os
REPO = 'someip-ids-multiclass-contentext'
if not os.path.exists('src/extract_ext.py') and not os.path.exists('data/ours_ext/X.npz'):
    if os.path.basename(os.getcwd()) == 'notebooks':
        os.chdir('..')
    elif os.path.isdir(REPO):
        os.chdir(REPO)
    else:
        os.system('apt-get -qq install -y git-lfs >/dev/null 2>&1; git lfs install')
        os.system(f'git clone -q https://github.com/GuilhermeFrick/{REPO}.git')
        os.chdir(REPO)
        os.system('git lfs pull')
if os.path.exists('data/ours_ext/X.npz') and os.path.getsize('data/ours_ext/X.npz') < 100000:
    os.system('apt-get -qq install -y git-lfs >/dev/null 2>&1; git lfs install; git lfs pull')
print('cwd:', os.getcwd())

FROM_SCRATCH = False   # True = baixa PCAPs + re-extrai do zero

In [ ]:
!pip -q install scapy xgboost scikit-learn matplotlib numpy pandas

In [ ]:
# 1) Download dos PCAPs (só se for do zero, ou se as features não existirem)
import os, glob
if FROM_SCRATCH or not os.path.exists('data/ours_ext/X.npz'):
    if len(glob.glob('data/pcap/*.pcap')) < 7:
        !python scripts/download_pcaps.py
    else:
        print('PCAPs já presentes em data/pcap/')
else:
    print('Features já presentes — pulando download. Use FROM_SCRATCH=True para refazer.')

In [ ]:
# 2) Extração de features (PCAP -> 21 features). ~6 min na primeira vez.
if FROM_SCRATCH or not os.path.exists('data/ours_ext/X.npz'):
    !python src/extract_ext.py --pcap-dir data/pcap --out data/ours_ext
else:
    print('Features já presentes — pulando extração.')

In [ ]:
# 3) Carrega features + content_ext (12 do Kim + 4 comportamentais, sem header)
import numpy as np, json
import matplotlib.pyplot as plt
from sklearn.metrics import (classification_report, confusion_matrix, roc_curve, auc,
                             precision_recall_curve, average_precision_score)
from sklearn.preprocessing import label_binarize
from xgboost import XGBClassifier

CLASSES = ['normal','dos','fuzzy','mitm_single','mitm_multi']; N=len(CLASSES)
CONTENT_EXT = list(range(12)) + [12, 13, 14, 16]
# contagem de pacotes por PCAP (ordem de extração) — usada no split temporal
FILE_COUNTS = [('benign',2193802),('dos',1864530),('fuzzy1',2197113),('fuzzy2',1304154),
               ('fuzzy3',2223650),('mitm_multi',2412529),('mitm_single',2037576)]

X = np.load('data/ours_ext/X.npz')['a'][:, CONTENT_EXT]
y = np.load('data/ours_ext/y_multi.npz')['a']
assert sum(c for _,c in FILE_COUNTS) == len(y), 'FILE_COUNTS não bate com os dados — reextração mudou as contagens?'
print('X:', X.shape, '| classes:', dict(zip(*[v.tolist() for v in np.unique(y, return_counts=True)])))

In [ ]:
# 4) SPLIT TEMPORAL POR ARQUIVO (correto): primeiros 70% / últimos 30% de cada PCAP
tr, te, start = [], [], 0
for _, cnt in FILE_COUNTS:
    cut = start + int(cnt * 0.7)
    tr.append(np.arange(start, cut)); te.append(np.arange(cut, start + cnt)); start += cnt
itr, ite = np.concatenate(tr), np.concatenate(te)
print(f'treino={len(itr):,}  teste={len(ite):,}')
print('teste — classes:', {CLASSES[c]: int((y[ite]==c).sum()) for c in range(N)})

clf = XGBClassifier(objective='multi:softprob', num_class=N, n_estimators=300, max_depth=8,
                    learning_rate=0.3, tree_method='hist', max_bin=256, n_jobs=-1, eval_metric='mlogloss')
clf.fit(X[itr], y[itr])
proba = clf.predict_proba(X[ite]); y_pred = proba.argmax(axis=1); y_te = y[ite]
print('treino concluído')

In [ ]:
# 5) Métricas por classe (split temporal — honesto)
print(classification_report(y_te, y_pred, target_names=CLASSES, digits=4))

In [ ]:
# 6) Matriz de confusão
cm = confusion_matrix(y_te, y_pred); cmn = cm / cm.sum(axis=1, keepdims=True)
fig, ax = plt.subplots(figsize=(7,6)); im = ax.imshow(cmn, cmap='Blues', vmin=0, vmax=1)
ax.set_xticks(range(N)); ax.set_yticks(range(N))
ax.set_xticklabels(CLASSES, rotation=45, ha='right'); ax.set_yticklabels(CLASSES)
ax.set_xlabel('Predito'); ax.set_ylabel('Verdadeiro'); ax.set_title('Matriz de Confusão — split temporal')
for i in range(N):
    for j in range(N):
        ax.text(j, i, f'{cmn[i,j]*100:.1f}%', ha='center', va='center', fontsize=8,
                color='white' if cmn[i,j]>0.5 else 'black')
fig.colorbar(im, fraction=0.046, pad=0.04); plt.tight_layout(); plt.show()

In [ ]:
# 7) Curvas ROC e Precisão-Recall (one-vs-rest)
yb = label_binarize(y_te, classes=range(N))
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14,6))
for i, name in enumerate(CLASSES):
    fpr, tpr, _ = roc_curve(yb[:,i], proba[:,i]); a = auc(fpr, tpr)
    s = max(len(fpr)//2000,1); ax1.plot(fpr[::s], tpr[::s], lw=1.6, label=f'{name} (AUC={a:.3f})')
    prec, rec, _ = precision_recall_curve(yb[:,i], proba[:,i]); ap = average_precision_score(yb[:,i], proba[:,i])
    s2 = max(len(rec)//2000,1); ax2.plot(rec[::s2], prec[::s2], lw=1.6, label=f'{name} (AP={ap:.3f})')
ax1.plot([0,1],[0,1],'k--',lw=0.7); ax1.set_xlabel('Falso Positivo'); ax1.set_ylabel('Verdadeiro Positivo')
ax1.set_title('ROC (one-vs-rest)'); ax1.legend(fontsize=8, loc='lower right')
ax2.set_xlabel('Recall'); ax2.set_ylabel('Precisão'); ax2.set_title('Precisão-Recall (one-vs-rest)')
ax2.legend(fontsize=8, loc='lower left'); plt.tight_layout(); plt.show()

## Leitura

- Split **temporal por arquivo** (sem vazamento) → macro-F1 ≈ **0,966** (vs 0,9936 do split
  aleatório otimista). Número **in-scope honesto**, nível de produção (> 0,95).
- A classe que mais cai é `mitm_multi` (relay, dependente de estrutura temporal).
- Comparação aleatório vs temporal: `02-comparacao-splits.ipynb`. Generalização a **ataque novo**
  (zero-day, ~0,60): `someip-ids-benchmark`.